In [1]:
import joblib
import pandas as pd

print("Carregando modelos...")
# Atualize o caminho se necessário
classificador = joblib.load('ArquivosPkl/classificador_fogo_cerrado.pkl')
regressor = joblib.load('ArquivosPkl/regressor_intensidade_frp.pkl')

# 1. Uma lista contendo vários cenários (Mocks) para testarmos de uma vez
lista_testes = [
    {
        "Nome": "O Dia Perfeito (Frio e Úmido)",
        "Dados": {"DiaSemChuva": 12, "Precipitacao": 5.2, "Temperatura_C": 22.5, "Umidade_Relativa_%": 78.0, "Vento_ms": 2.1, "Mes": 2, "Hora": 8, "Latitude": -19.92, "Longitude": -43.93}
    },
    {
        "Nome": "O Caos no Cerrado (Quente e Seco)",
        "Dados": {"DiaSemChuva": 95, "Precipitacao": 0.0, "Temperatura_C": 36.8, "Umidade_Relativa_%": 18.5, "Vento_ms": 8.4, "Mes": 9, "Hora": 14, "Latitude": -16.68, "Longitude": -49.25}
    },
    {
        "Nome": "O Falso Alarme (Quente, mas Choveu)",
        "Dados": {"DiaSemChuva": 1, "Precipitacao": 25.5, "Temperatura_C": 38.2, "Umidade_Relativa_%": 82.0, "Vento_ms": 3.1, "Mes": 1, "Hora": 15, "Latitude": -18.91, "Longitude": -48.27}
    },
    {
        "Nome": "O Perigo Silencioso (Frio, mas Ventando)",
        "Dados": {"DiaSemChuva": 85, "Precipitacao": 0.0, "Temperatura_C": 21.5, "Umidade_Relativa_%": 22.0, "Vento_ms": 14.5, "Mes": 7, "Hora": 13, "Latitude": -15.59, "Longitude": -47.33}
    },
    {
        "Nome": "teste",
        "Dados": {"DiaSemChuva": 35, "Precipitacao": 0.0, "Temperatura_C": 28.5, "Umidade_Relativa_%": 21.5, "Vento_ms": 2.0, "Mes": 6, "Hora": 21, "Latitude": -15.10, "Longitude": -44.01}
    }
    
]

limiar = 0.45

print("\n" + "="*50)
print("INICIANDO BATERIA DE TESTES DO MODELO")
print("="*50)

# 2. O Loop Mágico: Roda o modelo para cada cenário da lista
for teste in lista_testes:
    nome_cenario = teste["Nome"]
    dados_mockados = teste["Dados"]
    
    print(f"\n🧪 Testando Cenário: {nome_cenario}")
    
    # Transforma o JSON específico desta rodada em DataFrame
    df_entrada = pd.DataFrame([dados_mockados])

    # Aplicando a sua regra do teto de 120 dias
    df_entrada['DiaSemChuva'] = df_entrada['DiaSemChuva'].clip(upper=120)

    # 3. Passando a tabela no Classificador
    probabilidade = classificador.predict_proba(df_entrada)[0][1]
    
    # 4. Tomando a decisão
    if probabilidade >= limiar:
        print(f"   ► Probabilidade: {probabilidade * 100:.2f}%  -> STATUS: 🔴 Risco ALTO!")
        
        # Passando a MESMA tabela no Regressor para ver a força
        frp_estimado = regressor.predict(df_entrada)[0]
        frp_estimado = max(0.0, frp_estimado) # Evita números negativos
        
        print(f"   ► INTENSIDADE ESPERADA: 🔥 {frp_estimado:.2f} Megawatts")
    else:
        print(f"   ► Probabilidade: {probabilidade * 100:.2f}%  -> STATUS: 🟢 Risco BAIXO.")
        print("   ► INTENSIDADE ESPERADA: 0.00 Megawatts")

print("\n" + "="*50)
print("BATERIA DE TESTES FINALIZADA!")
print("="*50)

Carregando modelos...

INICIANDO BATERIA DE TESTES DO MODELO

🧪 Testando Cenário: O Dia Perfeito (Frio e Úmido)
   ► Probabilidade: 35.63%  -> STATUS: 🟢 Risco BAIXO.
   ► INTENSIDADE ESPERADA: 0.00 Megawatts

🧪 Testando Cenário: O Caos no Cerrado (Quente e Seco)
   ► Probabilidade: 100.00%  -> STATUS: 🔴 Risco ALTO!
   ► INTENSIDADE ESPERADA: 🔥 47.05 Megawatts

🧪 Testando Cenário: O Falso Alarme (Quente, mas Choveu)
   ► Probabilidade: 0.18%  -> STATUS: 🟢 Risco BAIXO.
   ► INTENSIDADE ESPERADA: 0.00 Megawatts

🧪 Testando Cenário: O Perigo Silencioso (Frio, mas Ventando)
   ► Probabilidade: 99.99%  -> STATUS: 🔴 Risco ALTO!
   ► INTENSIDADE ESPERADA: 🔥 34.80 Megawatts

🧪 Testando Cenário: teste
   ► Probabilidade: 100.00%  -> STATUS: 🔴 Risco ALTO!
   ► INTENSIDADE ESPERADA: 🔥 47.57 Megawatts

BATERIA DE TESTES FINALIZADA!
